# 03 - LoRA fine-tuning of IndicTrans2-200M (arm A)
Needs a **GPU runtime (T4)**. Run the cells in order; each says how long it takes.

1. login, code, environments (as in notebook 01)
2. training data: downloaded from your **private** Hugging Face repo, or built once and uploaded there
3. smoke test -> full training (seed 42) -> adapter uploaded to the private repo
4. translate FLORES with the adapter (en-hi, and en-mai for the Maithili transfer question) and score it
5. results printed into this notebook -> Ctrl+S -> tell Claude "done"

In [ ]:
# Cell 1: GPU check + Hugging Face login
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4")

from huggingface_hub import login, whoami
try:
    whoami()
    print("Already logged in to Hugging Face")
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face token: "))

In [ ]:
# Cell 2: get (or update) the code from GitHub and install packages
import os
REPO_URL = "https://github.com/prefierolasoledad/TourMT-Hi.git"
WORKDIR = "/content/tourmt-hi"

if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    # Make the Colab copy exactly match GitHub (GitHub is the master copy;
    # files changed only here on Colab are replaced by GitHub's version).
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} reset -q --hard origin/main
%cd {WORKDIR}
# Install packages; show only errors and the final summary (so failures are not hidden)
!pip install -r requirements.txt 2>&1 | grep -iE "error|failed|successfully installed" || true

# Check the packages we need actually import
import importlib
for pkg in ["transformers", "datasets", "sacrebleu"]:
    try:
        m = importlib.import_module(pkg)
        print(f"OK      {pkg} {getattr(m, '__version__', '')}")
    except Exception as e:
        print(f"MISSING {pkg}: {type(e).__name__}: {e}")
!git log -1 --oneline

In [ ]:
# Cell 2b: separate Python 3.11 environment for evaluation + IndicTrans2 (both need transformers<5)
# Takes 1-2 minutes; needed once per Colab session. See requirements-eval.txt.
import os
if not os.path.exists("/content/venv-eval/bin/python"):
    !pip install -q uv
    !uv venv -q /content/venv-eval --python 3.11
# Always run: installs anything new in requirements-eval.txt (fast if nothing changed)
!uv pip install -q --python /content/venv-eval/bin/python -r requirements-eval.txt
# --no-deps on purpose: its dependency is a broken fork of indicnlp (see requirements-eval.txt)
!uv pip install -q --python /content/venv-eval/bin/python --no-deps IndicTransToolkit==1.1.1
!/content/venv-eval/bin/python -c "import sys; sys.path.insert(0, 'src'); import comet, sacrebleu, torch, transformers, datasets; from IndicTransToolkit.processor import IndicProcessor; import peft; from common import check_indicnlp_normalizer; check_indicnlp_normalizer(); from importlib.metadata import version; print('eval env OK | comet', version('unbabel-comet'), '| transformers', transformers.__version__, '| peft', peft.__version__, '| GPU', torch.cuda.is_available())"

In [ ]:
# Cell 3: get the arm A training data (Colab deletes everything when the runtime resets, so the data
# is kept in a PRIVATE Hugging Face dataset repo "<you>/tourmt-hi-work" - only you can see it).
#   first time: builds it (about 1-1.5 hours on a T4: mostly reading 10M Samanantar pairs) and uploads it
#   later:      downloads it in seconds
import os, shutil
from huggingface_hub import HfApi, snapshot_download
api = HfApi()
WORK_REPO = f"{api.whoami()['name']}/tourmt-hi-work"
api.create_repo(WORK_REPO, repo_type="dataset", private=True, exist_ok=True)
ARM, FORCE_REBUILD = "arm_a", False
DATA_FILES = ["train.tsv", "dev.tsv", "filtered_before_alignment.json", "labse_scores.json"]
REPORT = f"results/finetune_data_{ARM}.json"

on_hub = f"finetune/{ARM}/train.tsv" in api.list_repo_files(WORK_REPO, repo_type="dataset")
if on_hub and not FORCE_REBUILD:
    local = snapshot_download(WORK_REPO, repo_type="dataset", allow_patterns=[f"finetune/{ARM}/*"])
    os.makedirs(f"data/finetune/{ARM}", exist_ok=True)
    for f in DATA_FILES:
        if os.path.exists(f"{local}/finetune/{ARM}/{f}"):
            shutil.copy(f"{local}/finetune/{ARM}/{f}", f"data/finetune/{ARM}/{f}")
    shutil.copy(f"{local}/finetune/{ARM}/finetune_data_{ARM}.json", REPORT)
    print("Downloaded the training data from", WORK_REPO)
else:
    !python src/build_finetune_data.py --config configs/finetune_{ARM}.yaml
    for f in DATA_FILES + [REPORT]:
        path = f if f == REPORT else f"data/finetune/{ARM}/{f}"
        if os.path.exists(path):
            api.upload_file(path_or_fileobj=path, path_in_repo=f"finetune/{ARM}/{os.path.basename(path)}",
                            repo_id=WORK_REPO, repo_type="dataset")
    print("Built and uploaded the training data to", WORK_REPO)
!wc -l data/finetune/{ARM}/train.tsv data/finetune/{ARM}/dev.tsv

In [ ]:
# Cell 4: smoke test - 30 training steps on 500 pairs (about 3 minutes incl. model download).
# Checks that everything works before the long run. Its files end in _smoke and are not kept.
PY = "/content/venv-eval/bin/python"
CFG = "configs/lora_it2-200m_arm-a_en-hi.yaml"
!{PY} src/train_lora.py --config {CFG} --max_steps 30 --limit 500

In [ ]:
# Cell 5: full training, seed 42 (about 20-40 minutes on a T4). Do NOT stop it while it is quiet:
# it prints the train loss every 50 steps and the dev loss every 200 steps.
SEED = 42
!{PY} src/train_lora.py --config {CFG} --seed {SEED}

In [ ]:
# Cell 6: upload the adapter (about 25 MB) and the training summary to your private repo
RUN = f"lora_it2-200m_arm-a_en-hi_s{SEED}"
api.upload_folder(folder_path=f"checkpoints/{RUN}/adapter", path_in_repo=f"adapters/{RUN}",
                  repo_id=WORK_REPO, repo_type="dataset")
api.upload_file(path_or_fileobj=f"results/train_{RUN}.json", path_in_repo=f"results/train_{RUN}.json",
                repo_id=WORK_REPO, repo_type="dataset")
print("Uploaded", RUN)

In [ ]:
# Cell 7: translate FLORES devtest with the adapter and score it (about 15 minutes)
#   en-hi : did fine-tuning on tourism data change general-domain quality? (RQ2: "small or no loss")
#   en-mai: does an English->Hindi adapter help Maithili? (RQ4 transfer)
if not os.path.exists(f"checkpoints/{RUN}/adapter"):   # e.g. after a runtime reset: get it from the repo
    local = snapshot_download(WORK_REPO, repo_type="dataset", allow_patterns=[f"adapters/{RUN}/*"])
    shutil.copytree(f"{local}/adapters/{RUN}", f"checkpoints/{RUN}/adapter")
for d in ["en-hi", "en-mai"]:
    !{PY} src/translate.py --config configs/lora-a-s{SEED}_flores-devtest_{d}.yaml
!{PY} src/evaluate.py --skip-scored --runs outputs/lora-a-s{SEED}_flores-devtest_en-hi outputs/lora-a-s{SEED}_flores-devtest_en-mai
!head -1 results/results.csv | cut -d, -f1,7-12 > /tmp/t.csv; grep -E "^(it2-200m|lora-a)" results/results.csv | grep -v hi-en | cut -d, -f1,7-12 >> /tmp/t.csv; column -s, -t /tmp/t.csv

In [ ]:
# Cell 8: put the results into this notebook's output, so they can be saved on the laptop (then Ctrl+S)
import base64, glob
!cd {WORKDIR} && rm -f /content/lora_results.zip && zip -qr /content/lora_results.zip outputs/lora-a-* results/results.csv results/finetune_data_*.json $(ls results/train_*.json | grep -v _smoke)
data = open("/content/lora_results.zip", "rb").read()
print(f"zip size: {len(data)/1024:.0f} KB")
print("===ZIP-START===")
print(base64.b64encode(data).decode())
print("===ZIP-END===")